In [10]:
import pandas as pd
import numpy as np

In [18]:
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

df_endereco_dados = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='iso-8859-1')

df_endereco_dados.columns = df_endereco_dados.columns.str.strip()

df_roubos_recuperacoes = df_endereco_dados[
    ['cisp', 'regiao', 'munic', 'roubo_veiculo', 'recuperacao_veiculos']
]

# corrigir Grande NiterÃÂÃÂÃÂÃÂ³i para Grande Niterói
df_roubos_recuperacoes = df_roubos_recuperacoes.replace('Grande NiterÃÂÃÂÃÂÃÂ³i', 'Grande Niterói')

# Agrupar por cisp regiao e munic e somar os valores de roubo_veiculo e recuperacao_veiculos
df_roubos_recuperacoes = df_roubos_recuperacoes.groupby(['cisp', 'regiao', 'munic'], as_index=False).sum()

# PROCESSAMENTO DOS DADOS

## Calculando metricas
roubo veiculo

In [19]:
# Calculo de quartis e IQR para roubo_veiculo
Q1_roubo = df_roubos_recuperacoes['roubo_veiculo'].quantile(0.25)
Q3_roubo = df_roubos_recuperacoes['roubo_veiculo'].quantile(0.75)
IQR_roubo = Q3_roubo - Q1_roubo

# Limites para identificar outliers em roubo_veiculo
limite_inferior_roubo = Q1_roubo - 1.5 * IQR_roubo
limite_superior_roubo = Q3_roubo + 1.5 * IQR_roubo

# Media e desvio padrao para roubo_veiculo
media_roubo = df_roubos_recuperacoes['roubo_veiculo'].mean()
desvio_padrao_roubo = df_roubos_recuperacoes['roubo_veiculo'].std()

max_roubo = df_roubos_recuperacoes['roubo_veiculo'].max()
min_roubo = df_roubos_recuperacoes['roubo_veiculo'].min()

recuperação veiculo

In [20]:
# Calculo de quartis e IQR para recuperacao_veiculos
Q1_recuperacao = df_roubos_recuperacoes['recuperacao_veiculos'].quantile(0.25)
Q3_recuperacao = df_roubos_recuperacoes['recuperacao_veiculos'].quantile(0.75)
IQR_recuperacao = Q3_recuperacao - Q1_recuperacao

# Limites para identificar outliers em recuperacao_veiculos
limite_inferior_recuperacao = Q1_recuperacao - 1.5 * IQR_recuperacao
limite_superior_recuperacao = Q3_recuperacao + 1.5 * IQR_recuperacao

# Media e desvio padrao para recuperacao_veiculos
media_recuperacao = df_roubos_recuperacoes['recuperacao_veiculos'].mean()
desvio_padrao_recuperacao = df_roubos_recuperacoes['recuperacao_veiculos'].std()

max_recuperacao = df_roubos_recuperacoes['recuperacao_veiculos'].max()
min_recuperacao = df_roubos_recuperacoes['recuperacao_veiculos'].min()

...


In [36]:
# Disponibiliza cisp, regiao e munic como colunas
df_base = df_roubos_recuperacoes.reset_index()

df_roubo_maiores = df_base[
    df_base['roubo_veiculo'] >= Q3_roubo
].copy()
df_roubo_maiores['flag'] = 'mais'

df_roubo_menores = df_base[
    df_base['roubo_veiculo'] <= Q1_roubo
].copy()
df_roubo_menores['flag'] = 'menos'


In [37]:
# concatenando os dois dataframes
df_roubo_flags = pd.concat([df_roubo_maiores, df_roubo_menores], ignore_index=True)

df_roubo_flags.to_csv('df_roubo_flags.csv', index=False, sep=';', encoding='utf-8-sig')